# 06 · Verificación en notebook limpio
Carga el modelo desde `models:/resenas-bancos-ec-modelo@champion` y comprueba que reproduce **exactamente** la métrica de prueba registrada en MLflow. Ejecutar en un kernel nuevo (*Restart* → *Run All*), después del `05`.

In [1]:
import mlflow, mlflow.sklearn, numpy as np
from sklearn.metrics import average_precision_score
from src.datos import cargar_obt
from src.pipeline import *
from src.experimento import NOMBRE_MODELO, configurar
configurar()
cli = mlflow.MlflowClient()
mv = cli.get_model_version_by_alias(NOMBRE_MODELO, "champion")
run = cli.get_run(mv.run_id)
modelo = mlflow.sklearn.load_model(f"models:/{NOMBRE_MODELO}@champion")
print("Modelo:", NOMBRE_MODELO, "| versión", mv.version, "| run_id", mv.run_id)

/usr/local/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm

Modelo: resenas-bancos-ec-modelo | versión 1 | run_id 1056d73032ae49bd83ab59977be1b7eb


In [2]:
df = agregar_features(cargar_obt())
train, test = split_temporal(df, 0.2)
p = modelo.predict_proba(test[FEATURES])[:, 1]
umbral = run.data.metrics["umbral_aplicado"]
pr_auc = average_precision_score(test[TARGET], p)
costo = costo_negocio(test[TARGET], (p >= umbral).astype(int))
print(f"PR-AUC prueba  → reproducida: {pr_auc:.6f} | registrada: {run.data.metrics['test_pr_auc']:.6f}")
print(f"Costo negocio  → reproducido: {costo:.6f} | registrado: {run.data.metrics['test_costo']:.6f}")
assert abs(pr_auc - run.data.metrics["test_pr_auc"]) < 1e-9 and abs(costo - run.data.metrics["test_costo"]) < 1e-9
print("OK: el modelo reproduce exactamente las métricas registradas.")

/usr/local/lib/python3.12/site-packages/snowflake/connector/connection/_connection.py:137: FutureWarning: enable_server_session_keep_alive_auto_detection was not set and defaults to True. In a future version, the default will change to None. Please provide an explicit value: True = check for running async queries before logout (queries are preserved); False/None = always send logout on close (async queries may be terminated by server). Logout behavior can also be overridden with server_session_keep_alive. See the connection parameter docs for more info.
  options = self.config.to_proto_options(


PR-AUC prueba  → reproducida: 0.890997 | registrada: 0.890997
Costo negocio  → reproducido: 0.170046 | registrado: 0.170046
OK: el modelo reproduce exactamente las métricas registradas.
